In [20]:
import sys
print(sys.version)
print(sys.executable)

3.11.0rc2 (main, Sep 11 2022, 20:22:52) [MSC v.1933 64 bit (AMD64)]
e:\tele_agent\Telethon\.venv\Scripts\python.exe


In [21]:
E:\Telethon\.venv\Scripts\activate
pip install ipykernel
python -m ipykernel install --user --name=telethon-venv --display-name "Signal Agent"

SyntaxError: unexpected character after line continuation character (1980909156.py, line 1)

In [ ]:
.venv\Scripts\activate

In [ ]:
##GET TELEGRAM CHANNEL ID

import os
from dotenv import load_dotenv
from telethon import TelegramClient

load_dotenv("E:/tele_agent/Telethon/.env")

api_id = int(os.getenv("TG_API_ID"))
api_hash = os.getenv("TG_API_HASH")

client = TelegramClient("session", api_id, api_hash)
await client.start()

async for dialog in client.iter_dialogs():
    print(dialog.id, dialog.name)

await client.disconnect()

-1003653629865 VASUKI BY BITSHARKS
-1002682649427 BITSHARKS
-1002951958446 Moby Mobile Product Group
777000 Telegram
-1001733042716 Tbvofficial
-1003561632635 TRIMP COMMUNITY
8660510290 Susanta Sbi Behala
8708167593 Rupali (Nirup)
8510359105 Deb Kumar HDFC
1623976336 lax mAn
8681313231 Alisha Georgina Lakra Work Num
8603145183 Sharwari
8318715096 Sharwari
8556279515 Hanut Singh Pgp 2023
8250247881 Ram Kaku
7499408528 BloFin Community Assistant


In [ ]:

pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.9 MB 11.7 MB/s eta 0:00:01
   ------------------- -------------------- 4.7/9.9 MB 11.4 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.9 MB 10.2 MB/s eta 0:00:01
   ----------------------------- ---------- 7.3/9.9 MB 8.9 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.9 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 8.3 MB/s  0:00:01
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/12.6 MB 8.4 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.6 MB 8.4 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.6 MB 8.6 MB/s eta 0:00:01
   ----------------------- ---------------- 7.3/12.6 MB 8.9 MB/s eta 0:00:01
   ----------------------------- ---------- 9.4/12.6 MB 9.0 MB/s eta 0:00:01
   -----------------

In [ ]:
import sqlite3
import pandas as pd
 
DB_PATH = r"E:\Tele_Agent\Telethon\data\signals.db"
 
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
 
# All tables in DB
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print("=== TABLES ===")
print(tables.to_string(index=False))
 
# Row counts
for t in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {t}", conn).iloc[0]['cnt']
    print(f"  {t}: {count:,} rows")

=== TABLES ===
                name
daily_signal_summary
       metrics_cache
             signals
   unparsed_messages
  daily_signal_summary: 837 rows
  metrics_cache: 0 rows
  signals: 9,731 rows
  unparsed_messages: 7 rows


In [ ]:
# =============================================================================
# CELL 1B — unparsed_messages: full audit
# Run after Cell 1 (conn already open)
# =============================================================================
 
print("=== unparsed_messages: row count ===")
print(pd.read_sql("SELECT COUNT(*) as total FROM unparsed_messages", conn).to_string(index=False))
 
print("\n=== reason distribution ===")
print(pd.read_sql("""
    SELECT reason, COUNT(*) as count,
           ROUND(COUNT(*)*100.0 / SUM(COUNT(*)) OVER(), 1) as pct
    FROM unparsed_messages
    GROUP BY reason
    ORDER BY count DESC
""", conn).to_string(index=False))
 
print("\n=== date spread of unparsed messages ===")
print(pd.read_sql("""
    SELECT date(timestamp) as day, COUNT(*) as count
    FROM unparsed_messages
    GROUP BY day
    ORDER BY day DESC
    LIMIT 20
""", conn).to_string(index=False))


# =============================================================================
# CELL 1C — unparsed_messages: raw text samples per reason
# Shows 3 examples per reason so you can see what's failing
# =============================================================================
 
reasons = pd.read_sql(
    "SELECT DISTINCT reason FROM unparsed_messages ORDER BY reason",
    conn
)['reason'].tolist()
 
for reason in reasons:
    print(f"\n{'='*60}")
    print(f"REASON: {reason}")
    print('='*60)
    samples = pd.read_sql("""
        SELECT message_id, timestamp, sender_id, raw_text
        FROM unparsed_messages
        WHERE reason = ?
        ORDER BY timestamp DESC
        LIMIT 3
    """, conn, params=(reason,))
    for _, row in samples.iterrows():
        print(f"\n  msg_id={row['message_id']}  ts={row['timestamp']}  sender={row['sender_id']}")
        print(f"  ---")
        print(f"  {repr(row['raw_text'][:300])}")
 

=== unparsed_messages: row count ===
 total
     7

=== reason distribution ===
    reason  count  pct
 NO_TICKER      5 71.4
EMPTY_TEXT      2 28.6

=== date spread of unparsed messages ===
       day  count
2026-04-09      1
2026-04-08      6

REASON: EMPTY_TEXT

  msg_id=1106  ts=2026-04-09 03:21:51.000000  sender=-1003653629865
  ---
  ''

  msg_id=1  ts=2026-04-08 07:28:35.000000  sender=-1003653629865
  ---
  ''

REASON: NO_TICKER

  msg_id=45  ts=2026-04-08 09:41:16.000000  sender=-1003653629865
  ---
  '🛑 **BitSharks stopped.**'

  msg_id=13  ts=2026-04-08 08:24:49.000000  sender=-1003653629865
  ---
  '🛑 **BitSharks stopped.**'

  msg_id=12  ts=2026-04-08 08:06:23.000000  sender=-1003653629865
  ---
  '🛑 **BitSharks stopped.**'


In [ ]:
# CELL 2 — signals: timestamp format inspection
# =============================================================================
sample = pd.read_sql("""
    SELECT message_id, ticker, timestamp, created_at
    FROM signals
    ORDER BY message_id ASC
    LIMIT 10
""", conn)
 
print("=== TIMESTAMP SAMPLE (first 10 rows) ===")
print(sample.to_string(index=False))
 
# Check raw stored type
raw = conn.execute("SELECT typeof(timestamp), timestamp FROM signals LIMIT 5").fetchall()
print("\n=== TYPEOF(timestamp) ===")
for r in raw:
    print(f"  type={r[0]}  value={r[1]}")
 
# Date range
date_range = pd.read_sql("""
    SELECT
        MIN(timestamp) as earliest,
        MAX(timestamp) as latest,
        COUNT(DISTINCT date(timestamp)) as distinct_days
    FROM signals
""", conn)
print("\n=== DATE RANGE ===")
print(date_range.to_string(index=False))
 

=== TIMESTAMP SAMPLE (first 10 rows) ===
 message_id  ticker                  timestamp                 created_at
          3    ALGO 2026-04-08 08:00:27.000000 2026-04-14 11:15:18.522519
          4     JOE 2026-04-08 08:00:31.000000 2026-04-14 11:15:18.512742
          5 VIRTUAL 2026-04-08 08:00:35.000000 2026-04-14 11:15:18.497853
          6  KERNEL 2026-04-08 08:00:39.000000 2026-04-14 11:15:18.489075
          7    DEXE 2026-04-08 08:01:46.000000 2026-04-14 11:15:18.477238
          8    AEVO 2026-04-08 08:01:50.000000 2026-04-14 11:15:18.468017
          9   KAITO 2026-04-08 08:01:53.000000 2026-04-14 11:15:18.458020
         11     XPL 2026-04-08 08:06:08.000000 2026-04-14 11:15:18.438716
         14    OPEN 2026-04-08 08:31:12.000000 2026-04-14 11:15:18.423290
         15    OPEN 2026-04-08 08:32:20.000000 2026-04-14 11:15:18.413291

=== TYPEOF(timestamp) ===
  type=text  value=2026-04-08 08:00:27.000000
  type=text  value=2026-04-08 08:00:31.000000
  type=text  value=2026-04

In [ ]:
# =============================================================================
# CELL 3 — signals: full column null audit
# =============================================================================
cols = pd.read_sql("PRAGMA table_info(signals)", conn)['name'].tolist()
print("=== NULL AUDIT — signals ===")
print(f"{'Column':<25} {'Total':>8} {'Nulls':>8} {'Null%':>8} {'Distinct':>10}")
print("-" * 62)
 
total = pd.read_sql("SELECT COUNT(*) as n FROM signals", conn).iloc[0]['n']
for col in cols:
    nulls = pd.read_sql(f"SELECT COUNT(*) as n FROM signals WHERE {col} IS NULL", conn).iloc[0]['n']
    distinct = pd.read_sql(f"SELECT COUNT(DISTINCT {col}) as n FROM signals WHERE {col} IS NOT NULL", conn).iloc[0]['n']
    pct = (nulls / total * 100) if total > 0 else 0
    flag = " ⚠️" if pct > 10 else ""
    print(f"{col:<25} {total:>8,} {nulls:>8,} {pct:>7.1f}%{flag:>3} {distinct:>10,}")

=== NULL AUDIT — signals ===
Column                       Total    Nulls    Null%   Distinct
--------------------------------------------------------------
message_id                   9,733        0     0.0%         9,733
ticker                       9,733        0     0.0%           288
pair                         9,733        0     0.0%           288
price_at_signal              9,733        0     0.0%         4,920
change_24h                   9,733        0     0.0%         1,063
activity_raw                 9,733        0     0.0%             8
activity_type                9,733        0     0.0%             2
boost                        9,733        0     0.0%            10
alerts_this_hour             9,733        0     0.0%            49
alerts_tier                  9,733        0     0.0%             3
has_media                    9,733        0     0.0%             1
sender_id                    9,733        0     0.0%             1
timestamp                    9,733      

In [ ]:
# =============================================================================
# CELL 4 — signals: activity_type + alerts_tier distributions
# =============================================================================
print("=== activity_type distribution ===")
act = pd.read_sql("""
    SELECT activity_type, COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct
    FROM signals
    GROUP BY activity_type
    ORDER BY count DESC
""", conn)
print(act.to_string(index=False))
 
print("\n=== alerts_tier distribution ===")
tier = pd.read_sql("""
    SELECT alerts_tier, COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct
    FROM signals
    GROUP BY alerts_tier
    ORDER BY count DESC
""", conn)
print(tier.to_string(index=False))
 
print("\n=== boost distribution ===")
boost = pd.read_sql("""
    SELECT boost, COUNT(*) as count
    FROM signals
    GROUP BY boost
    ORDER BY boost
""", conn)
print(boost.to_string(index=False))
 

=== activity_type distribution ===
activity_type  count  pct
          BUY   5872 60.3
         SELL   3866 39.7

=== alerts_tier distribution ===
alerts_tier  count  pct
     NORMAL   4875 50.1
        HOT   2517 25.8
       FIRE   2346 24.1

=== boost distribution ===
 boost  count
     1     16
     2    150
     3    465
     4    780
     5   1355
     6   1950
     7   2833
     8   1700
     9    440
    10     49


In [22]:
# =============================================================================
# CELL 5 — signals: rows on/after April 14 (excluded from daily_calls backfill)
# =============================================================================
 
#NWe'll check both UTC date and ET date to understand the boundary
future = pd.read_sql("""
    SELECT
        date(timestamp) as utc_date,
        COUNT(*) as count
    FROM signals
    WHERE date(timestamp) >= '2026-04-14'
    GROUP BY utc_date
    ORDER BY utc_date
""", conn)
print("=== Rows on/after 2026-04-14 (UTC) ===")
print(future.to_string(index=False) if len(future) > 0 else "  None — all data is before cutoff ✅")
 
# ET offset check — currently EDT = UTC-4
future_et = pd.read_sql("""
    SELECT
        date(timestamp, '-4 hours') as et_date,
        COUNT(*) as count
    FROM signals
    WHERE date(timestamp, '-4 hours') >= '2026-04-14'
    GROUP BY et_date
    ORDER BY et_date
""", conn)
print("\n=== Rows on/after 2026-04-14 (ET = UTC-4) ===")
print(future_et.to_string(index=False) if len(future_et) > 0 else "  None ✅")
 

=== Rows on/after 2026-04-14 (UTC) ===
  utc_date  count
2026-04-14   1033

=== Rows on/after 2026-04-14 (ET = UTC-4) ===
   et_date  count
2026-04-14    727


In [23]:
# =============================================================================
# CELL 6 — metrics_cache: coverage vs signals tickers
# =============================================================================
coverage = pd.read_sql("""
    SELECT
        COUNT(DISTINCT s.ticker) as total_signal_tickers,
        COUNT(DISTINCT m.ticker) as cached_tickers,
        COUNT(DISTINCT CASE WHEN m.ticker IS NULL THEN s.ticker END) as uncached_tickers,
        ROUND(COUNT(DISTINCT m.ticker) * 100.0 / COUNT(DISTINCT s.ticker), 1) as coverage_pct
    FROM signals s
    LEFT JOIN metrics_cache m ON m.ticker = s.ticker
""", conn)
print("=== metrics_cache coverage ===")
print(coverage.to_string(index=False))
 
# Null audit on metrics_cache
print("\n=== NULL AUDIT — metrics_cache ===")
mcols = pd.read_sql("PRAGMA table_info(metrics_cache)", conn)['name'].tolist()
mtotal = pd.read_sql("SELECT COUNT(*) as n FROM metrics_cache", conn).iloc[0]['n']
for col in mcols:
    nulls = pd.read_sql(f"SELECT COUNT(*) as n FROM metrics_cache WHERE {col} IS NULL", conn).iloc[0]['n']
    pct = (nulls / mtotal * 100) if mtotal > 0 else 0
    flag = " ⚠️" if pct > 50 else ""
    print(f"  {col:<20} nulls={nulls:>6,}  ({pct:.1f}%){flag}")
 
# MCap distribution for what we have
print("\n=== MCap distribution (existing cache) ===")
mcap_dist = pd.read_sql("""
    SELECT
        CASE
            WHEN mcap < 50000000          THEN 'micro  (<50M)'
            WHEN mcap < 500000000         THEN 'small  (50M-500M)'
            WHEN mcap < 5000000000        THEN 'mid    (500M-5B)'
            WHEN mcap >= 5000000000       THEN 'large  (>5B)'
            ELSE 'unknown (null mcap)'
        END as tier,
        COUNT(*) as count
    FROM metrics_cache
    GROUP BY tier
    ORDER BY count DESC
""", conn)
print(mcap_dist.to_string(index=False))
 

=== metrics_cache coverage ===
 total_signal_tickers  cached_tickers  uncached_tickers  coverage_pct
                  289               0               289           0.0

=== NULL AUDIT — metrics_cache ===
  ticker               nulls=     0  (0.0%)
  price                nulls=     0  (0.0%)
  volume_24h           nulls=     0  (0.0%)
  mcap                 nulls=     0  (0.0%)
  rank                 nulls=     0  (0.0%)
  circ_supply          nulls=     0  (0.0%)
  fetched_at           nulls=     0  (0.0%)

=== MCap distribution (existing cache) ===
Empty DataFrame
Columns: [tier, count]
Index: []


In [24]:
# =============================================================================
# CELL 7 — DRY RUN: daily_calls candidate preview (NO WRITES)
#  Using UTC-4 offset for ET day. Confirm DST offset from Cell 2 output first.
# Change '-4 hours' to '-5 hours' if timestamps are during EST (Nov-Mar).
# =============================================================================
 
dry_run = pd.read_sql("""
    SELECT
        ticker,
        date(timestamp, '-4 hours')                                         AS et_day,
        activity_type,
 
        -- first call
        MIN(message_id)                                                     AS first_call_msg_id,
        MIN(timestamp)                                                      AS first_call_ts_utc,
        time(MIN(timestamp), '-4 hours')                                    AS first_call_time_et,
 
        -- last call (will equal first if call_count=1)
        MAX(message_id)                                                     AS last_call_msg_id,
        MAX(timestamp)                                                      AS last_call_ts_utc,
        time(MAX(timestamp), '-4 hours')                                    AS last_call_time_et,
 
        COUNT(*)                                                            AS call_count,
        MAX(boost)                                                          AS max_boost
 
    FROM signals
    WHERE activity_type IN ('BUY', 'SELL')
      AND date(timestamp, '-4 hours') < '2026-04-14'
    GROUP BY ticker, date(timestamp, '-4 hours'), activity_type
    ORDER BY et_day DESC, call_count DESC
    LIMIT 50
""", conn)
 
print(f"=== DRY RUN: daily_calls preview (top 50 rows by call_count) ===")
print(f"Projected total rows: ", end="")
total_proj = pd.read_sql("""
    SELECT COUNT(*) as n FROM (
        SELECT ticker, date(timestamp, '-4 hours') as et_day, activity_type
        FROM signals
        WHERE activity_type IN ('BUY', 'SELL')
          AND date(timestamp, '-4 hours') < '2026-04-14'
        GROUP BY ticker, date(timestamp, '-4 hours'), activity_type
    )
""", conn).iloc[0]['n']
print(f"{total_proj:,}")
 
# Data quality flags
print(f"\nRows where first_call = last_call (only 1 call that day): checking...")
single = pd.read_sql("""
    SELECT COUNT(*) as n FROM (
        SELECT ticker, date(timestamp, '-4 hours') as d, activity_type
        FROM signals
        WHERE activity_type IN ('BUY', 'SELL')
          AND date(timestamp, '-4 hours') < '2026-04-14'
        GROUP BY ticker, d, activity_type
        HAVING COUNT(*) = 1
    )
""", conn).iloc[0]['n']
print(f"  Single-call days: {single:,} ({single/total_proj*100:.1f}%) — first=last, drift=0%")
 
print(dry_run.to_string(index=False))
 

=== DRY RUN: daily_calls preview (top 50 rows by call_count) ===
Projected total rows: 1,201

Rows where first_call = last_call (only 1 call that day): checking...
  Single-call days: 322 (26.8%) — first=last, drift=0%
     ticker     et_day activity_type  first_call_msg_id          first_call_ts_utc first_call_time_et  last_call_msg_id           last_call_ts_utc last_call_time_et  call_count  max_boost
BROCCOLI714 2026-04-13           BUY               7301 2026-04-13 08:30:44.000000           04:30:44              8888 2026-04-14 02:11:56.000000          22:11:56          50         10
     GIGGLE 2026-04-13           BUY               7051 2026-04-13 04:01:32.000000           00:01:32              8704 2026-04-13 23:55:19.000000          19:55:19          46         10
        TST 2026-04-13           BUY               7239 2026-04-13 07:35:17.000000           03:35:17              8079 2026-04-13 18:36:43.000000          14:36:43          44          9
       HOLO 2026-04-13       

In [25]:
# =============================================================================
# CELL 8 — price_at_signal quality check for first/last calls
# =============================================================================
 
price_check = pd.read_sql("""
    WITH grouped AS (
        SELECT
            ticker,
            date(timestamp, '-4 hours') as et_day,
            activity_type,
            MIN(message_id) as first_msg_id,
            MAX(message_id) as last_msg_id
        FROM signals
        WHERE activity_type IN ('BUY', 'SELL')
          AND date(timestamp, '-4 hours') < '2026-04-14'
        GROUP BY ticker, date(timestamp, '-4 hours'), activity_type
    )
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN s1.price_at_signal IS NULL THEN 1 ELSE 0 END) as first_price_null,
        SUM(CASE WHEN s2.price_at_signal IS NULL THEN 1 ELSE 0 END) as last_price_null,
        ROUND(AVG(CASE WHEN s1.price_at_signal IS NULL THEN 1.0 ELSE 0.0 END) * 100, 1) as first_null_pct,
        ROUND(AVG(CASE WHEN s2.price_at_signal IS NULL THEN 1.0 ELSE 0.0 END) * 100, 1) as last_null_pct
    FROM grouped g
    JOIN signals s1 ON s1.message_id = g.first_msg_id
    JOIN signals s2 ON s2.message_id = g.last_msg_id
""", conn)
 
print("=== price_at_signal quality for first + last calls ===")
print(price_check.to_string(index=False))
 

=== price_at_signal quality for first + last calls ===
 total_rows  first_price_null  last_price_null  first_null_pct  last_null_pct
       1201                 0                0             0.0            0.0


In [26]:
# =============================================================================
# CELL 9 — CREATE daily_calls table (DDL only, no data yet)
# Run this ONCE. Safe to re-run — uses IF NOT EXISTS.
# =============================================================================
 
conn.execute("""
CREATE TABLE IF NOT EXISTS daily_calls (
    -- Primary key
    id                          INTEGER PRIMARY KEY AUTOINCREMENT,
 
    -- Dimensions
    ticker                      TEXT    NOT NULL,
    et_day                      TEXT    NOT NULL,   -- YYYY-MM-DD in ET
    activity_type               TEXT    NOT NULL,   -- BUY | SELL
 
    -- First call anchor
    first_call_msg_id           INTEGER NOT NULL,
    first_call_price            REAL,
    first_call_time_et          TEXT,               -- HH:MM:SS
 
    -- Last call anchor
    last_call_msg_id            INTEGER NOT NULL,
    last_call_price             REAL,
    last_call_time_et           TEXT,
 
    -- Day aggregates
    call_count                  INTEGER NOT NULL DEFAULT 1,
    max_boost                   INTEGER,
 
    -- EOD price — fetched separately via CoinGecko, null until filled
    eod_price                   REAL,
    eod_fetched_at              TEXT,               -- ISO UTC datetime
 
    -- MCap — snapshotted from metrics_cache at backfill time
    mcap_at_call                REAL,
    mcap_tier                   TEXT,               -- micro | small | mid | large | unknown
 
    -- Computed efficiency (Python fills these after eod_price arrives)
    first_call_efficiency_pct   REAL,               -- (eod - first_price) / first_price * 100
    last_call_efficiency_pct    REAL,               -- (eod - last_price) / last_price * 100
    intraday_drift_pct          REAL,               -- (last_price - first_price) / first_price * 100
    avg_trade_hours             REAL,               -- hours from first_call_time_et to 23:59 ET
    direction_correct           INTEGER,            -- 1 | 0 | NULL (bool, null if eod missing)
 
    -- Data quality flags (bitmask via text for readability)
    dq_first_price_missing      INTEGER DEFAULT 0,
    dq_last_price_missing       INTEGER DEFAULT 0,
    dq_eod_missing              INTEGER DEFAULT 1,  -- starts as 1, cleared when eod fetched
    dq_mcap_missing             INTEGER DEFAULT 0,
 
    -- Unique constraint: one row per (ticker, day, direction)
    UNIQUE(ticker, et_day, activity_type)
)
""")
conn.commit()
print("✅ daily_calls table created (or already exists)")
 
# Verify
verify = pd.read_sql("PRAGMA table_info(daily_calls)", conn)
print(f"\n{len(verify)} columns defined:")
print(verify[['name', 'type', 'notnull', 'dflt_value']].to_string(index=False))
 

✅ daily_calls table created (or already exists)

25 columns defined:
                     name    type  notnull dflt_value
                       id INTEGER        0        NaN
                   ticker    TEXT        1        NaN
                   et_day    TEXT        1        NaN
            activity_type    TEXT        1        NaN
        first_call_msg_id INTEGER        1        NaN
         first_call_price    REAL        0        NaN
       first_call_time_et    TEXT        0        NaN
         last_call_msg_id INTEGER        1        NaN
          last_call_price    REAL        0        NaN
        last_call_time_et    TEXT        0        NaN
               call_count INTEGER        1          1
                max_boost INTEGER        0        NaN
                eod_price    REAL        0        NaN
           eod_fetched_at    TEXT        0        NaN
             mcap_at_call    REAL        0        NaN
                mcap_tier    TEXT        0        NaN
first_call_ef

In [27]:
# =============================================================================
# CELL 10 — BACKFILL INSERT into daily_calls from signals
# Adjust '-4 hours' offset based on Cell 2 findings.
# This INSERT OR IGNORE — safe to re-run (won't duplicate).
# EOD + efficiency fields left NULL — filled by separate Python fetcher job.
# =============================================================================
 
conn.execute("""
INSERT OR IGNORE INTO daily_calls (
    ticker, et_day, activity_type,
    first_call_msg_id, first_call_price, first_call_time_et,
    last_call_msg_id,  last_call_price,  last_call_time_et,
    call_count, max_boost,
    mcap_at_call, mcap_tier,
    intraday_drift_pct, avg_trade_hours,
    dq_first_price_missing, dq_last_price_missing, dq_mcap_missing
)
SELECT
    s_grp.ticker,
    s_grp.et_day,
    s_grp.activity_type,
 
    -- first call
    s_first.message_id,
    s_first.price_at_signal,
    time(s_first.timestamp, '-4 hours'),
 
    -- last call
    s_last.message_id,
    s_last.price_at_signal,
    time(s_last.timestamp, '-4 hours'),
 
    s_grp.call_count,
    s_grp.max_boost,
 
    -- mcap from cache at time of backfill
    m.mcap,
    CASE
        WHEN m.mcap IS NULL             THEN 'unknown'
        WHEN m.mcap < 50000000          THEN 'micro'
        WHEN m.mcap < 500000000         THEN 'small'
        WHEN m.mcap < 5000000000        THEN 'mid'
        ELSE                                 'large'
    END,
 
    -- intraday drift (null if either price missing)
    CASE
        WHEN s_first.price_at_signal IS NOT NULL
         AND s_last.price_at_signal  IS NOT NULL
         AND s_first.price_at_signal > 0
        THEN ROUND((s_last.price_at_signal - s_first.price_at_signal)
                   / s_first.price_at_signal * 100.0, 4)
        ELSE NULL
    END,
 
    -- avg_trade_hours: hours from first call to 23:59 ET
    CASE
        WHEN s_first.price_at_signal IS NOT NULL
        THEN ROUND(
            (strftime('%s', s_grp.et_day || ' 23:59:00')
             - strftime('%s', datetime(s_first.timestamp, '-4 hours')))
            / 3600.0, 2)
        ELSE NULL
    END,
 
    -- data quality flags
    CASE WHEN s_first.price_at_signal IS NULL THEN 1 ELSE 0 END,
    CASE WHEN s_last.price_at_signal  IS NULL THEN 1 ELSE 0 END,
    CASE WHEN m.mcap                  IS NULL THEN 1 ELSE 0 END
 
FROM (
    SELECT
        ticker,
        date(timestamp, '-4 hours')     AS et_day,
        activity_type,
        MIN(message_id)                 AS first_msg_id,
        MAX(message_id)                 AS last_msg_id,
        COUNT(*)                        AS call_count,
        MAX(boost)                      AS max_boost
    FROM signals
    WHERE activity_type IN ('BUY', 'SELL')
      AND date(timestamp, '-4 hours') < '2026-04-14'
    GROUP BY ticker, date(timestamp, '-4 hours'), activity_type
) s_grp
JOIN signals s_first ON s_first.message_id = s_grp.first_msg_id
JOIN signals s_last  ON s_last.message_id  = s_grp.last_msg_id
LEFT JOIN metrics_cache m ON m.ticker = s_grp.ticker
""")
conn.commit()
 
inserted = pd.read_sql("SELECT COUNT(*) as n FROM daily_calls", conn).iloc[0]['n']
print(f"✅ Backfill complete — {inserted:,} rows in daily_calls")
 
 

✅ Backfill complete — 1,201 rows in daily_calls


In [28]:
# =============================================================================
# CELL 11 — VALIDATION: daily_calls post-backfill checks
# =============================================================================
 
print("=== Row count ===")
print(pd.read_sql("SELECT COUNT(*) as total FROM daily_calls", conn).to_string(index=False))
 
print("\n=== activity_type split ===")
print(pd.read_sql("""
    SELECT activity_type, COUNT(*) as rows,
           ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) as pct
    FROM daily_calls GROUP BY activity_type
""", conn).to_string(index=False))
 
print("\n=== mcap_tier distribution ===")
print(pd.read_sql("""
    SELECT mcap_tier, COUNT(*) as rows
    FROM daily_calls GROUP BY mcap_tier ORDER BY rows DESC
""", conn).to_string(index=False))
 
print("\n=== data quality flag summary ===")
print(pd.read_sql("""
    SELECT
        SUM(dq_first_price_missing) as first_price_missing,
        SUM(dq_last_price_missing)  as last_price_missing,
        SUM(dq_eod_missing)         as eod_missing,
        SUM(dq_mcap_missing)        as mcap_missing,
        COUNT(*)                    as total
    FROM daily_calls
""", conn).to_string(index=False))
 
print("\n=== call_count distribution ===")
print(pd.read_sql("""
    SELECT call_count, COUNT(*) as days
    FROM daily_calls
    GROUP BY call_count
    ORDER BY call_count
    LIMIT 20
""", conn).to_string(index=False))
 
print("\n=== intraday_drift_pct stats ===")
print(pd.read_sql("""
    SELECT
        ROUND(MIN(intraday_drift_pct), 2) as min_drift,
        ROUND(MAX(intraday_drift_pct), 2) as max_drift,
        ROUND(AVG(intraday_drift_pct), 2) as avg_drift,
        COUNT(intraday_drift_pct)         as non_null_count
    FROM daily_calls
""", conn).to_string(index=False))
 
print("\n=== Sample rows (10 most recent BUY days) ===")
print(pd.read_sql("""
    SELECT ticker, et_day, activity_type, call_count, max_boost,
           first_call_price, last_call_price, intraday_drift_pct,
           avg_trade_hours, mcap_tier,
           dq_first_price_missing, dq_eod_missing
    FROM daily_calls
    WHERE activity_type = 'BUY'
    ORDER BY et_day DESC
    LIMIT 10
""", conn).to_string(index=False))
 
 


=== Row count ===
 total
  1201

=== activity_type split ===
activity_type  rows  pct
          BUY   679 56.5
         SELL   522 43.5

=== mcap_tier distribution ===
mcap_tier  rows
  unknown  1201

=== data quality flag summary ===
 first_price_missing  last_price_missing  eod_missing  mcap_missing  total
                   0                   0         1201          1201   1201

=== call_count distribution ===
 call_count  days
          1   322
          2   163
          3   115
          4    95
          5    61
          6    53
          7    47
          8    31
          9    29
         10    34
         11    26
         12    18
         13    22
         14    16
         15    15
         16    15
         17    10
         18     9
         19     9
         20     9

=== intraday_drift_pct stats ===
 min_drift  max_drift  avg_drift  non_null_count
    -52.05      50.82        1.4            1195

=== Sample rows (10 most recent BUY days) ===
  ticker     et_day activ

In [29]:
# =============================================================================
# CELL 12 — Hourly analysis preview: which hour fires fresh first calls most?
# =============================================================================
 
print("=== Hourly first-call distribution (BUY + SELL combined, ET) ===")
hourly = pd.read_sql("""
    SELECT
        SUBSTR(first_call_time_et, 1, 2)    AS hour_et,
        COUNT(*)                             AS total_first_calls,
        SUM(CASE WHEN activity_type='BUY'  THEN 1 ELSE 0 END) AS buy_calls,
        SUM(CASE WHEN activity_type='SELL' THEN 1 ELSE 0 END) AS sell_calls,
        ROUND(AVG(call_count), 2)            AS avg_calls_per_ticker,
        ROUND(AVG(max_boost), 2)             AS avg_max_boost
    FROM daily_calls
    GROUP BY hour_et
    ORDER BY total_first_calls DESC
""", conn)
print(hourly.to_string(index=False))
 
conn.close()
print("\n✅ All checks complete. Share output and we proceed to EOD fetcher + efficiency computation.")

=== Hourly first-call distribution (BUY + SELL combined, ET) ===
hour_et  total_first_calls  buy_calls  sell_calls  avg_calls_per_ticker  avg_max_boost
     10                 87         51          36                  8.06           6.78
     00                 86         48          38                 12.23           7.13
     08                 85         50          35                  4.87           6.89
     11                 68         42          26                  8.26           7.32
     04                 67         35          32                  7.63           6.76
     09                 64         36          28                  7.98           7.08
     18                 61         32          29                  4.67           7.03
     02                 60         34          26                 13.35           7.58
     01                 55         29          26                  7.36           6.82
     06                 54         29          25                

In [32]:
# Quick status check — did daily_calls get created and populated?
import sqlite3, pandas as pd

DB_PATH = r"E:\tele_Agent\Telethon\data\signals.db"
conn = sqlite3.connect(DB_PATH)

# Check table exists
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print("Tables:", tables['name'].tolist())

# If daily_calls exists, check row count and date range
try:
    summary = pd.read_sql("""
        SELECT 
            COUNT(*) as total_rows,
            MIN(et_day) as earliest_day,
            MAX(et_day) as latest_day,
            SUM(CASE WHEN activity_type='BUY' THEN 1 ELSE 0 END) as buy_rows,
            SUM(CASE WHEN activity_type='SELL' THEN 1 ELSE 0 END) as sell_rows,
            SUM(dq_eod_missing) as eod_missing
        FROM daily_calls
    """, conn)
    print("\ndaily_calls status:")
    print(summary.to_string(index=False))
except Exception as e:
    print(f"\ndaily_calls does NOT exist yet: {e}")

conn.close()

Tables: ['daily_calls', 'daily_signal_summary', 'metrics_cache', 'signals', 'sqlite_sequence', 'unparsed_messages']

daily_calls status:
 total_rows earliest_day latest_day  buy_rows  sell_rows  eod_missing
       1201   2026-04-08 2026-04-13       679        522         1201


In [5]:
import sys, os
sys.path.insert(0, r"E:\tele_agent\Telethon")
os.chdir(r"E:\tele_agent\Telethon")

from dotenv import load_dotenv
load_dotenv(r"E:\tele_agent\Telethon\.env")

from src.backfill_eod import run_eod_backfill

for day in ["2026-04-8", "2026-04-9", "2026-04-10", "2026-04-11", "2026-04-12"]:
    result = run_eod_backfill(day)
    print(result)

{'et_day': '2026-04-8', 'updated': 0, 'failed': 0, 'skipped_no_price': 0, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-9', 'updated': 0, 'failed': 0, 'skipped_no_price': 0, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-10', 'updated': 185, 'failed': 0, 'skipped_no_price': 25, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-11', 'updated': 183, 'failed': 0, 'skipped_no_price': 11, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-12', 'updated': 168, 'failed': 0, 'skipped_no_price': 20, 'skipped_dq_first_missing': 0}


In [8]:
# In notebook — rerun backfill for days that have eod_price already filled
from src.backfill_eod import run_eod_backfill

for day in ["2026-04-13", "2026-04-14", "2026-04-15", "2026-04-16"]:
    result = run_eod_backfill(day)
    print(result)

{'et_day': '2026-04-13', 'updated': 214, 'failed': 0, 'skipped_no_price': 40, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-14', 'updated': 183, 'failed': 0, 'skipped_no_price': 28, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-15', 'updated': 149, 'failed': 0, 'skipped_no_price': 22, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-16', 'updated': 265, 'failed': 0, 'skipped_no_price': 50, 'skipped_dq_first_missing': 0}


In [7]:
from src.backfill_eod import run_eod_backfill

for day in ["2026-04-7","2026-04-8", "2026-04-9", "2026-04-10", "2026-04-11", "2026-04-12"]:
    result = run_eod_backfill(day)
    print(result)

{'et_day': '2026-04-7', 'updated': 0, 'failed': 0, 'skipped_no_price': 0, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-8', 'updated': 0, 'failed': 0, 'skipped_no_price': 0, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-9', 'updated': 0, 'failed': 0, 'skipped_no_price': 0, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-10', 'updated': 177, 'failed': 0, 'skipped_no_price': 33, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-11', 'updated': 175, 'failed': 0, 'skipped_no_price': 19, 'skipped_dq_first_missing': 0}
{'et_day': '2026-04-12', 'updated': 150, 'failed': 0, 'skipped_no_price': 38, 'skipped_dq_first_missing': 0}
